# Optimizer, Dropout, and TTA

- using dropout in regressor
- using AdamW with weight_decay
- targets transform using log1p and expm1
- targets are known to be somewhat dependents. So, this notebook only use 'Dry_Clover_g', 'Dry_Dead_g', 'Dry_Green_g',
  - 'Dry_Total_g' is  'Dry_Clover_g' + 'Dry_Dead_g' + 'Dry_Green_g'
  - 'GDM_g'is 'Dry_Clover_g' + 'Dry_Green_g'

## Imports

In [ ]:
!pip install /kaggle/input/pip-show-protobuf/protobuf-3.20.3-py2.py3-none-any.whl

In [ ]:
import os
import re
import glob
import time

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import cv2

from PIL import Image
from tqdm import tqdm

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split


import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Subset, Dataset

from transformers import AutoImageProcessor, AutoModel

In [ ]:
img_dir_path = "/kaggle/input/csiro-biomass"

In [ ]:
print("Read csv data")
d_train = pd.read_csv("/kaggle/input/csiro-biomass/train.csv", parse_dates=["Sampling_Date"])
d_test = pd.read_csv("/kaggle/input/csiro-biomass/test.csv")
print("Done")

## Analysis

In [ ]:
data = []
for idx, row in d_train.iterrows():
    img = Image.open(os.path.join(img_dir_path, row.image_path))
    w, h = img.size
    data.append({'path': row.image_path, 'w': w, 'h': h})

In [ ]:
d_size = pd.DataFrame(data)

In [ ]:
d_size.w.value_counts(), d_size.h.value_counts()

In [ ]:
sns.boxplot(d_train, x='target', y='target_name')
plt.show()

In [ ]:
d_img_metrics = d_train.groupby(by=['image_path', 'Species'], as_index=False)[['Pre_GSHH_NDVI', 'Height_Ave_cm']].max()

In [ ]:
sns.boxplot(d_img_metrics, x='Pre_GSHH_NDVI', y='Species')

In [ ]:
sns.histplot(d_train.loc[d_train.target_name == 'Dry_Total_g', 'target'])

In [ ]:
d_temp = d_train.pivot(index='image_path', columns='target_name', values='target').reset_index()
fig, ax = plt.subplots(1, 3, figsize=(10, 5))
sns.scatterplot(d_temp, x='Dry_Clover_g', y='Dry_Total_g', ax=ax[0])
sns.scatterplot(d_temp, x='Dry_Dead_g', y='Dry_Total_g', ax=ax[1])
sns.scatterplot(d_temp, x='Dry_Green_g', y='Dry_Total_g', ax=ax[2])
plt.tight_layout()

In [ ]:
d_temp.columns

In [ ]:
sns.heatmap(d_temp.iloc[:, 1:].corr(), annot=True, fmt='.2f')

In [ ]:
d_temp['clover_to_total_ratio'] = d_temp.Dry_Clover_g / d_temp.Dry_Total_g
d_temp['dead_to_total_ratio'] = d_temp.Dry_Dead_g / d_temp.Dry_Total_g
d_temp['green_to_total_ratio'] = d_temp.Dry_Green_g / d_temp.Dry_Total_g
d_temp['clover_to_gdm_ratio'] = d_temp.Dry_Clover_g / d_temp.GDM_g
d_temp['green_to_gdm_ratio'] = d_temp.Dry_Green_g / d_temp.GDM_g

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(10, 20))
counter = 0
for idx, row in d_temp.sort_values('clover_to_total_ratio', ascending=False).head(5).iterrows():
    im_arr = cv2.imread(os.path.join(img_dir_path, row.image_path))
    axes[counter].imshow(im_arr)
    axes[counter].axis('off')
    axes[counter].set_title(f"Clover: {row.Dry_Clover_g} Dead: {row.Dry_Dead_g} Green: {row.Dry_Green_g}")
    counter += 1

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(10, 20))
counter = 0
for idx, row in d_temp.sort_values('dead_to_total_ratio', ascending=False).head(5).iterrows():
    im_arr = cv2.imread(os.path.join(img_dir_path, row.image_path))
    axes[counter].imshow(im_arr)
    axes[counter].axis('off')
    axes[counter].set_title(f"Clover: {row.Dry_Clover_g} Dead: {row.Dry_Dead_g} Green: {row.Dry_Green_g}")
    counter += 1

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(10, 20))
counter = 0
for idx, row in d_temp.sort_values('green_to_total_ratio', ascending=False).head(5).iterrows():
    im_arr = cv2.imread(os.path.join(img_dir_path, row.image_path))
    axes[counter].imshow(im_arr)
    axes[counter].axis('off')
    axes[counter].set_title(f"Clover: {row.Dry_Clover_g} Dead: {row.Dry_Dead_g} Green: {row.Dry_Green_g}")
    counter += 1

## Load DINOv2 backbone

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

In [ ]:
print("Load dinov2 model")
# processor = AutoImageProcessor.from_pretrained('facebook/dinov2-giant')
processor = AutoImageProcessor.from_pretrained('/kaggle/input/dinov2/pytorch/giant/1')
model = AutoModel.from_pretrained('/kaggle/input/dinov2/pytorch/giant/1')
model = model.to(device)
model = model.eval()
print("Done")

#### Get embedding from DINOv2

In [ ]:
def extract_embedding_from_model(img_path):
    img_full_path = os.path.join(img_dir_path, img_path)
    img = Image.open(img_full_path)
    w, h = img.size
    img = img.resize((int(w/4), int(h/4)))
    inputs = processor(images=img, return_tensors="pt")
    inputs = inputs.to(device)
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    mean_feat = outputs.last_hidden_state[:, 1:, :].mean(dim=1)
    
    return mean_feat.squeeze(0)

In [ ]:
start = time.time()
data_img_embed = {}
for path in tqdm(d_train.image_path.unique()):
    img_embedding = extract_embedding_from_model(path)
    data_img_embed[path] = img_embedding
    break

duration = time.time() - start
print(f"Get image embedding duration: {duration:.2f}s")

img_emb_size = img_embedding.cpu().numpy().shape[0]
print(f"Img Emb size: {img_emb_size}")

## Prepare Dataset

In [ ]:
d_train_pivot = d_train.pivot(index='image_path', columns='target_name', values='target').reset_index()
d_train_pivot.columns.name = None

In [ ]:
d_train_pivot.columns

In [ ]:
d_train_selected = d_train_pivot[['image_path', 'Dry_Clover_g', 'Dry_Dead_g', 'Dry_Green_g']].copy()

In [ ]:
train, valid = train_test_split(d_train_selected, test_size=0.2, random_state=123)
train.shape, valid.shape

In [ ]:
def apply_ss(train, valid):
    
    ss_map = {}
    for col in train.columns[1:]:
        ss_map[col] = StandardScaler()
        target_train_scaled = ss_map[col].fit_transform(train[col].to_numpy().reshape(-1, 1))
        valid_train_scaled = ss_map[col].transform(valid[col].to_numpy().reshape(-1, 1))
        train[col] = target_train_scaled
        valid[col] = valid_train_scaled

    return train, valid

In [ ]:
def apply_log_transform(train, valid):
    train.iloc[:, 1:] = train.iloc[:, 1:].apply(np.log1p)
    valid.iloc[:, 1:] = valid.iloc[:, 1:].apply(np.log1p)

    return train, valid

In [ ]:
log_transform = True
if log_transform:
    train, valid = apply_log_transform(train, valid)

In [ ]:
train.head()

In [ ]:
valid.head()

In [ ]:
id2target = dict((idx, col) for idx, col in enumerate(train.columns[1:]))
target2id = dict((col, idx) for idx, col in enumerate(train.columns[1:]))

In [ ]:
class CSIRO(Dataset):
    def __init__(self, train, valid):
        self.split = {
            'train': (train, len(train)),
            'valid': (valid, len(valid))
        }

        self.set_split(split="train")

    def set_split(self, split='train'):
        self.data, self.length = self.split[split]

    def __getitem__(self, idx):
        img_path = self.data.iloc[idx, 0]
        x = extract_embedding_from_model(img_path)
         
        y = np.array(self.data.iloc[idx, 1:].tolist())

        return x, y

    def __len__(self):
        return self.length

In [ ]:
dataset = CSIRO(train, valid)
data_gen = DataLoader(dataset, batch_size = 2)
x, y = next(iter(data_gen))
x = x.to(device)
y = y.to(device)
print("x.shape and y.shape after data loader", x.shape, y.shape)

## Regressor

In [ ]:
class MLP(nn.Module):
    def __init__(self, emb_size):
        super(MLP, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(emb_size, 512),
            nn.LeakyReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 256),
            nn.LeakyReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 3),
            nn.ReLU(),
        )

    def forward(self, input_x):
        out = self.model(input_x)

        return out

In [ ]:
print("Create MLP model...")
model_regressor = MLP(img_emb_size)
model_regressor = model_regressor.to(device)

total_params = sum(param.numel() for param in model_regressor.parameters())
print(f"Total regressor params: {total_params:,}")
print("Done")

## Loss Function

In [ ]:
class WeightedMSELoss(nn.Module):
    def __init__(self):
        super(WeightedMSELoss, self).__init__()
        # Competition weights
        self.weights = torch.tensor([0.1, 0.1, 0.1, 0.5, 0.2])  # [Dry_Clover, Dry_Dead, Dry_Green, Dry_Total, GDM]
    
    def forward(self, pred, target):
        # pred, target shape: [batch_size, 5]
        mse = (pred - target) ** 2
        weighted_mse = mse * self.weights.to(pred.device)
        
        return weighted_mse.mean()

In [ ]:
class WeightedHuberLoss(nn.Module):
    def __init__(self, delta=30):
        super().__init__()
        self.delta = delta
        self.weights = torch.tensor([0.3, 0.3, 0.4]).to(device) # [Dry_Clover, Dry_Dead, Dry_Green, Dry_Total, GDM]
        
    def forward(self, pred, target):
        error = pred - target
        is_small = torch.abs(error) <= self.delta
        small_loss = 0.5 * error ** 2
        large_loss = self.delta * torch.abs(error) - 0.5 * self.delta ** 2
        
        # Weight by target magnitude
        loss = torch.where(is_small, small_loss, large_loss)
        weighted_loss = loss * self.weights
        
        return weighted_loss.mean()

In [ ]:
def compute_weighted_r2(pred, target):
    weights = torch.Tensor([0.3, 0.3, 0.4]).to(device)
    
    # Compute weighted R²
    # weighted_mean = (weights * target).mean()
    # ss_res = torch.sum(weights * (target - pred)**2)
    # ss_tot = torch.sum(weights * (target - weighted_mean)**2)
    # score = 1 - (ss_res / ss_tot)

    # Correction
    col_means = target.mean(dim=0)
    ss_res = ((target - pred) ** 2).sum(dim=0)
    ss_tot = ((target - col_means) ** 2).sum(dim=0)
    r2_per_col = 1 - (ss_res / ss_tot)
    score = (weights * r2_per_col).sum()
    
    return score

## Training routine

In [ ]:
print("Define optimizer and loss function...")
optimizer = optim.AdamW(model_regressor.parameters(), lr=0.0005, weight_decay=0.02)
if log_transform:
    criterion = WeightedHuberLoss(delta=1.0)
else:
    criterion = WeightedHuberLoss(delta=30)
print(f"Optimizer: {optimizer.__class__.__name__}")
print(f"Loss function: {criterion.__class__.__name__}")
print("Done")

In [ ]:
for epoch in range(1, 101):
    start = time.time()

    train_loss = 0
    valid_loss = 0
    train_r2 = 0
    valid_r2 = 0
    
    dataset.set_split("train")
    data_gen = DataLoader(dataset, batch_size=512, shuffle=False)
    model_regressor.train()
    for batch_index, (x, y) in enumerate(data_gen, 1):
        x = x.to(device)
        y = y.to(device)
        
        model_regressor.zero_grad()

        out = model_regressor(x)
        loss = criterion(out, y)
        
        train_loss += (loss.item() - train_loss) / batch_index

        r2 = compute_weighted_r2(out, y)
        train_r2 += (r2 - train_r2) / batch_index

        loss.backward()
        optimizer.step()

    dataset.set_split("valid")
    data_gen = DataLoader(dataset, batch_size=512, shuffle=False)
    model_regressor.eval()
    for batch_index, (x, y) in enumerate(data_gen, 1):
        x = x.to(device)
        y = y.to(device)
        
        with torch.no_grad():
            out = model_regressor(x)

        loss = criterion(out, y)

        valid_loss += (loss.item() - valid_loss) / batch_index

        r2 = compute_weighted_r2(out, y)
        valid_r2 += (r2 - valid_r2) / batch_index

    duration = int(time.time() - start)
    print(f"Epoch: {epoch} | Time: {duration}s")
    print(f"\t Train loss: {train_loss} | R2: {train_r2}")
    print(f"\t Valid loss: {valid_loss} | R2: {valid_r2}")

## Sanity Check

In [ ]:
start = time.time()
result_inference = []
model_regressor.eval()
for path in d_train_pivot.head(5).image_path.unique():

    embedding = extract_embedding_from_model(path)

    with torch.no_grad():
        y_pred = model_regressor(embedding)

    clover, dead, green = y_pred
    
    result_inference.append({
        'image_path': path, 
        'Dry_Clover_g': np.expm1(clover.item()),
        'Dry_Dead_g': np.expm1(dead.item()), 
        'Dry_Green_g': np.expm1(green.item()),
        'Dry_Total_g': np.expm1(clover.item() + dead.item() + green.item()), 
        'GDM_g': np.expm1(clover.item() + green.item())
    })
    
duration = time.time() - start
print(f"Regressor duration on test: {duration:.2f}s")

In [ ]:
d_train_sc = pd.DataFrame(result_inference)

In [ ]:
pd.concat((d_train_pivot.set_index('image_path'), d_train_sc.set_index('image_path')), axis=1, join='inner')

## Submission

In [ ]:
d_test.head()

In [ ]:
d_test.shape

In [ ]:
start = time.time()
result_inference = []
model_regressor.eval()
for path in d_test.image_path.unique():

    embedding = extract_embedding_from_model(path)

    with torch.no_grad():
        y_pred = model_regressor(embedding)

    clover, dead, green = y_pred
    clover, dead, green = np.expm1(clover.item()), np.expm1(dead.item()), np.expm1(green.item()) 
    
    result_inference.append({
        'image_path': path, 
        'Dry_Clover_g': clover, 
        'Dry_Dead_g': dead, 
        'Dry_Green_g': green, 
        'Dry_Total_g': clover + dead + green, 
        'GDM_g': clover + green
    })

duration = time.time() - start
print(f"Regressor duration on test: {duration:.2f}s")

In [ ]:
d_test_pred_raw = pd.DataFrame(result_inference)
target_names = ["Dry_Clover_g", "Dry_Dead_g", "Dry_Green_g", "Dry_Total_g", "GDM_g"]
d_test_pred = d_test_pred_raw.melt(id_vars='image_path', value_vars=target_names)
d_test_pred = d_test_pred.rename({'variable': 'target_name', 'value': 'target'}, axis=1)

In [ ]:
d_test_pred.head()

In [ ]:
d_submission = pd.merge(left=d_test, right=d_test_pred, how='left', on=['image_path', 'target_name'])
d_submission[['sample_id', 'target']].to_csv('submission.csv', index=False)
d_submission